In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/sample_submission.csv
/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/train.csv
/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/metadata.csv
/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/test.csv


In [4]:
# --------------- 1. LOAD DATA --------------------------------
train = pd.read_csv('/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/train.csv', low_memory=False)
test  = pd.read_csv('/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/test.csv',  low_memory=False)

# MILESTONE 1
## Question 1 (MCQ): Dataset Dimensions 
Before building any models, it is crucial to understand the scale of the data. How many records (rows) are present in the train.csv and test.csv datasets, respectively?

A) Train: 100,000 | Test: 25,000

#### B) Train: 138,701 | Test: 15,000

C) Train: 150,000 | Test: 10,000

D) Train: 138,701 | Test: 25,000

## Question 2 (MCQ): Target Variable Identification 
By doing a strict set difference between the columns in train.csv and test.csv, which column is correctly identified as the sole target variable we need to predict?

A) OperationalHoursMeter

#### B) TargetValue

C) TransactionID

D) AssetID

## Question 3 (Numeric): Feature Data Types
 In the train dataset, how many columns are strictly recognized as numerical features (int64 or float64) by default?

A) 12

B) 10

#### C) 6

D) 8

## Question 4 (MSQ): Extremely Sparse Columns 
Which of the following configuration columns exhibit a missing value rate higher than 90% (missing over 124,800 rows out of 138,701) in the training set and should likely be dropped or heavily grouped? (Select all that apply)

#### A) col18

#### B) col19

C) CabinType

#### D) col5

## Question 5 (Numeric): Missing Operational Metrics 
The OperationalHoursMeter represents the lifetime active runtime of the machine. What is the approximate percentage of missing values for this critical column in the training dataset?

#### 40.838205924975306

## Question 6 (MCQ): Target Value Distribution 
Heavy machinery prices are often right-skewed. What is the median of TargetValue (transaction price) in the dataset?

A) $41,521

B) $30,000

#### C) $35,000

D) $50,000

## Question 7 (Numeric): Manufacture Year Anomaly
 When analyzing the ManufactureYear, you will find a strange anomaly. What is the most frequently occurring (mode) ManufactureYear in the dataset, indicating a default placeholder value used by the data entry system?

#### 1001
 
## Question 8 (MCQ): Time Span of Transactions 
What is the earliest (minimum) TransactionDate recorded in the training dataset?

#### A) 1990-01-31

B) 1989-05-12

C) 2000-01-01

D) 1995-12-15

## Question 9 (Numeric): High Cardinality Identification
 Machine learning models struggle with high-cardinality categorical features. How many unique string classes exist within the Spec_BaseClass column?
 #### 1249
 
## Question 10 (MCQ): Geographical Volume 
Which region accounts for the highest volume of machinery transactions (RegionCode) in this dataset?

A) Texas

B) California

#### C) Florida

D) North Carolina

## Question 11 (Numeric): Regional Pricing Insights 
For the most frequent region that is identified as Florida, what is the approximate average TargetValue for machines sold there?
#### 45007.54262574595

## Question 12 (MCQ): Asset Utilization 
The UtilizationTier metadata column ranks how heavily an asset was used. Based on the value counts in the training set, what is the most common utilization tier?

A) High

#### B) Medium

C) Low

D) Unspecified

## Question 13 (MCQ): Operational Correlation

 Intuitively, one might assume that machines with higher OperationalHoursMeter values would sell for less. What is the Pearson correlation coefficient between TargetValue and OperationalHoursMeter?

A) -0.45 (Strong Negative)

B) -0.15 (Weak Negative)

#### C) ~0.002 (Virtually Zero)

D) +0.20 (Weak Positive)

## Question 14 (MCQ): Primary Equipment Types 
Based on the FunctionalClassification column, what is the description of the single most common exact machinery traded on this platform?

A) Wheel Loader - 150.0 to 175.0 Horsepower

#### B) Hydraulic Excavator, Track - 21.0 to 24.0 Metric Tons

C) Skid Steer Loader - 75.0 to 90.0 Horsepower

D) Track Excavators - Medium

In [22]:
print("Q-1")
print(train.shape,test.shape)
print("="*40)

print('Q-2')
print(set(train.columns) - set(test.columns))
print("="*40)

print("Q-3")
print(train.info())
print("there are 6 colums")
print("="*40)

print("Q-4")
print((train.isnull().mean()*100).sort_values(ascending=False))
print("="*60)

print("Q-5")
print(train['OperationalHoursMeter'].isnull().mean() * 100)
print("="*60)

print("Q-6")
print(train['TargetValue'].median())
print("="*60)

print("Q-7")
print(train['ManufactureYear'].unique())
print(train['ManufactureYear'].mode()[0])
print("="*60)

print("Q-8")
print(train['TransactionDate'].nunique())
print(train['TransactionDate'].min())

print("Q-9")
print("Unique classes in Spec_BaseClass:")
print(train['Spec_BaseClass'].nunique())

print("="*60)
print("Q-10")
print("\nMost frequent region:")
print(train['RegionCode'].mode()[0])

print("="*60)
print("Q-11")
florida_avg = train.loc[
    train['RegionCode'].astype(str).str.contains('Florida', case=False, na=False),
    'TargetValue'
].mean()
print("Average TargetValue in Florida:")
print(florida_avg)

print("="*60)
print("Q-12")
print("UtilizationTier counts:")
print(train['UtilizationTier'].value_counts(dropna=False))
print("\nMost common UtilizationTier:")
print(train['UtilizationTier'].mode()[0])

print("="*60)
print("Q-13")
corr = train['TargetValue'].corr(
    train['OperationalHoursMeter']
)
print("Pearson Correlation:")
print(corr)

print("="*60)
print("Q-14")
print("FunctionalClassification counts:")
print(train['FunctionalClassification'].value_counts())
print("\nMost common FunctionalClassification:")
print(train['FunctionalClassification'].mode()[0])



Q-1
(138701, 50) (15000, 49)
Q-2
{'TargetValue'}
Q-3
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 138701 entries, 0 to 138700
Data columns (total 50 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   TransactionID              138701 non-null  int64  
 1   TargetValue                138701 non-null  float64
 2   AssetID                    138701 non-null  int64  
 3   ProductConfigID            138701 non-null  int64  
 4   DataOriginCode             138701 non-null  object 
 5   VendorPartnerID            138701 non-null  object 
 6   ManufactureYear            138701 non-null  int64  
 7   OperationalHoursMeter      82058 non-null   float64
 8   UtilizationTier            50475 non-null   object 
 9   TransactionDate            138701 non-null  object 
 10  Spec_FullDescriptor        138701 non-null  object 
 11  Spec_BaseClass             138701 non-null  object 
 12  Spec_SubClass              101716